# 천안시 생활시설 통합 및 지오코딩

5개 생활지수 폴더의 원본 CSV를 표준 형식으로 통합한 뒤, 위도·경도가 결측인 시설만 GIMI9 API로 지오코딩합니다.

- 원본 CSV 통합 → 투영좌표 변환 → 결측 좌표 지오코딩 → 결과 검증을 한 번에 수행합니다.
- API 토큰은 환경변수 `GIMI9_API_TOKEN` 또는 실행 중 입력으로 받습니다.
- 기존에 좌표가 있는 행은 수정하지 않습니다.

In [1]:
from pathlib import Path
from getpass import getpass
import math
import os
import re
import struct
import time
import zipfile

import numpy as np
import pandas as pd
import requests
from pyproj import Transformer

IS_COLAB = False
try:
    import google.colab  # type: ignore
    from google.colab import drive  # type: ignore
    drive.mount('/content/drive')
    IS_COLAB = True
except ModuleNotFoundError:
    pass

if IS_COLAB:
    PROJECT_DIR = Path('/content/drive/MyDrive/Cheonan')
else:
    PROJECT_DIR = Path.cwd()

ROOT = PROJECT_DIR / 'grid_inputs'
INTEGRATED_PATH = ROOT / 'cheonan_all_facilities_integrated.csv'
OUTPUT_PATH = ROOT / 'cheonan_all_facilities_geocoded.csv'
BOUNDARY_ZIP = ROOT / 'cheonan_adm_dong_boundary_20250630.zip'
INDEX_FOLDERS = ['life', 'commerce', 'medical', 'education', 'leisure']
SOURCE_EXTENSIONS = {'.csv', '.xlsx', '.xls'}
OUTPUT_COLUMNS = ['생활지수', '대분류', '중분류', 'name', '도로명주소', '지번주소', '위도', '경도', 'source_file']

if not ROOT.exists():
    raise FileNotFoundError(f'입력 폴더가 없습니다: {ROOT}')
if not BOUNDARY_ZIP.exists():
    raise FileNotFoundError(f'천안시 경계 ZIP이 없습니다: {BOUNDARY_ZIP}')

# 쇼핑몰과 편의점의 투영좌표를 WGS84 경위도로 변환합니다.
TRANSFORMERS = {
    'shoppingmall': Transformer.from_crs('EPSG:5174', 'EPSG:4326', always_xy=True),
    'convstore': Transformer.from_crs('EPSG:3857', 'EPSG:4326', always_xy=True),
    # 공연장 XLSX의 좌표정보(X/Y)는 쇼핑몰과 동일하게 EPSG:5174로 변환합니다.
    'live_hall': Transformer.from_crs('EPSG:5174', 'EPSG:4326', always_xy=True),
}
LON_RANGE = (127.0, 127.5)
LAT_RANGE = (36.5, 37.0)
API_URL = 'https://geocode-api.gimi9.com/geocode'
def normalize_api_token(value):
    token = str(value or '').strip().strip('\"').strip("'")
    token = re.sub(r'^(authorization\s*:\s*|bearer\s+)', '', token, flags=re.IGNORECASE)
    return token.strip()

API_TOKEN = normalize_api_token(os.environ.get('GIMI9_API_TOKEN', ''))
if not API_TOKEN:
    API_TOKEN = normalize_api_token(getpass('GIMI9 API 토큰을 입력하세요 (화면에 표시되지 않음): '))
if not API_TOKEN:
    raise RuntimeError('GIMI9 API 토큰이 없습니다.')
BATCH_SIZE = 500
REQUEST_INTERVAL_SECONDS = 0.2
print(f'실행 환경: {"Google Colab" if IS_COLAB else "로컬"}')
print('프로젝트 폴더:', PROJECT_DIR)
print('입력 폴더:', ROOT)
print('최종 출력:', OUTPUT_PATH)

GIMI9 API 토큰을 입력하세요 (화면에 표시되지 않음):  ········


실행 환경: 로컬
프로젝트 폴더: C:\Users\심현석\Documents\test\Cheonan-0825
입력 폴더: C:\Users\심현석\Documents\test\Cheonan-0825\grid_inputs
최종 출력: C:\Users\심현석\Documents\test\Cheonan-0825\grid_inputs\cheonan_all_facilities_geocoded.csv


## 1. 원본 시설 CSV 통합 및 좌표 변환

In [2]:
def read_csv_with_encoding(path):
    last_error = None
    for encoding in ('utf-8-sig', 'utf-8', 'cp949', 'euc-kr'):
        try:
            return pd.read_csv(path, encoding=encoding), encoding
        except Exception as exc:
            last_error = exc
    raise RuntimeError(f'CSV를 읽을 수 없습니다: {path} / {last_error}')

def read_tabular_with_metadata(path):
    if path.suffix.lower() == '.csv':
        frame, encoding = read_csv_with_encoding(path)
        return frame, f'csv:{encoding}'
    if path.suffix.lower() in {'.xlsx', '.xls'}:
        try:
            return pd.read_excel(path), f'excel:{path.suffix.lower()}'
        except ImportError as exc:
            raise ImportError(
                'XLSX 파일을 읽으려면 openpyxl 패키지가 필요합니다. '
                'Colab에서는 !pip install openpyxl 후 다시 실행하세요.'
            ) from exc
    raise ValueError(f'지원하지 않는 입력 형식입니다: {path}')

def clean_text(value):
    if pd.isna(value):
        return pd.NA
    value = str(value).strip()
    return value if value else pd.NA

def col(df, *names):
    for name in names:
        if name in df.columns:
            return df[name]
    return pd.Series(pd.NA, index=df.index, dtype='object')

def normalize_header(value):
    if pd.isna(value):
        return ''
    return re.sub(r'\s+', '', str(value)).strip()

SOURCE_MAJOR_ALIASES = {
    'concert': 'live_hall',
    'sportandresort': 'sportsandresort',
}
MAJOR_GROUP_BY_SOURCE = {
    'market': 'commercial_area',
    'shoppingmall': 'commercial_area',
    'museum': 'cultural_area',
    'sportsandresort': 'cultural_area',
    'library': 'cultural_area',
    'theater': 'cultural_area',
    'live_hall': 'cultural_area',
    'cultural_facility': 'cultural_area',
}
SCHOOL_MAJOR_BY_LEVEL = {
    '유치원': 'school_younger',
    '초등학교': 'school_younger',
    '특수학교': 'school_younger',
    '중학교': 'school_older',
    '고등학교': 'school_older',
}

def source_major_from_filename(path):
    remainder = path.stem.split('_', 1)[1]
    major = remainder[:-len('_cheonan')] if remainder.endswith('_cheonan') else remainder
    return SOURCE_MAJOR_ALIASES.get(major, major)

def major_from_filename(path):
    return MAJOR_GROUP_BY_SOURCE.get(source_major_from_filename(path), source_major_from_filename(path))

def middle_category(major, df):
    # 통합 대분류로 묶이는 시설은 원자료의 파일명을 중분류로 유지합니다.
    merged_middle_labels = {
        'market': 'market',
        'shoppingmall': 'shoppingmall',
        'museum': 'museum',
        'sportsandresort': 'sportandresort',
        'library': 'library',
        'theater': 'theater',
        'live_hall': 'live_hall',
        'cultural_facility': 'cultural_facility',
    }
    if major in merged_middle_labels:
        return pd.Series([merged_middle_labels[major]] * len(df), index=df.index)
    if major == 'agency':
        values = col(df, '기관명').fillna('').astype(str)
        return pd.Series(['시청구청' if ('시청' in x or '구청' in x) else '행정복지센터' if '행정복지센터' in x else '기타' for x in values], index=df.index)
    if major == 'park':
        return col(df, '공원구분').map(clean_text)
    if major == 'hospital':
        return col(df, '종별코드명').map(clean_text)
    if major == 'school':
        return col(df, '학교급').map(clean_text)
    fixed = {
        'convstore': '편의점', 'pharmacy': '약국',
        'welfare-facilities-for-elderly': '노인복지시설',
    }
    if major not in fixed:
        raise ValueError(f'정의되지 않은 대분류입니다: {major}')
    return pd.Series([fixed[major]] * len(df), index=df.index)

NAME = {
    'agency': ('기관명',), 'park': ('공원명',), 'welfare-facilities-for-elderly': (' 기관명', '기관명'),
    'market': ('시장명',), 'shoppingmall': ('사업장명',), 'convstore': ('fclty_nm',),
    'hospital': ('요양기관명',), 'pharmacy': ('요양기관명',), 'school': ('학교명',),
    'library': ('도서관명',), 'museum': ('시설명',), 'sportsandresort': ('명칭',), 'theater': ('사업장명',),
    'live_hall': ('사업장명',),
    'cultural_facility': ('시설명',),
}
ROAD = {
    'agency': ('주소',), 'park': ('소재지도로명주소',), 'welfare-facilities-for-elderly': (' 주소', '주소'),
    'market': ('소재지도로명주소',), 'shoppingmall': ('도로명주소',), 'convstore': ('rn_adres',),
    'hospital': ('주소',), 'pharmacy': ('주소',), 'school': ('주소',), 'library': ('소재지도로명주소',),
    'museum': ('소재지도로명주소',), 'sportsandresort': ('주소',), 'theater': ('도로명전체주소',),
    'live_hall': ('도로명주소',),
    'cultural_facility': ('주소',),
}
PARCEL = {
    'park': ('소재지지번주소',), 'market': ('소재지지번주소',), 'shoppingmall': ('지번주소',),
    'convstore': ('adres',), 'theater': ('소재지전체주소',), 'live_hall': ('지번주소',),
}
LAT = {'park': ('위도',), 'market': ('위도',), 'hospital': ('좌표(Y)',), 'pharmacy': ('좌표(Y)',), 'library': ('위도',), 'museum': ('위도',)}
LON = {'park': ('경도',), 'market': ('경도',), 'hospital': ('좌표(X)',), 'pharmacy': ('좌표(X)',), 'library': ('경도',), 'museum': ('경도',)}

PROJECTED_COORD_COLUMNS = {
    'shoppingmall': ('좌표정보(X)', '좌표정보(Y)'),
    'live_hall': ('좌표정보(X)', '좌표정보(Y)'),
}

def projected_to_wgs84(df, major):
    x_name, y_name = PROJECTED_COORD_COLUMNS.get(major, ('x', 'y'))
    x = pd.to_numeric(col(df, x_name), errors='coerce')
    y = pd.to_numeric(col(df, y_name), errors='coerce')
    lon = pd.Series(np.nan, index=df.index, dtype=float)
    lat = pd.Series(np.nan, index=df.index, dtype=float)
    valid = x.notna() & y.notna()
    if valid.any():
        try:
            out_lon, out_lat = TRANSFORMERS[major].transform(x.loc[valid].to_numpy(), y.loc[valid].to_numpy())
            out_lon, out_lat = np.asarray(out_lon), np.asarray(out_lat)
            ok = np.isfinite(out_lon) & np.isfinite(out_lat) & (out_lon >= LON_RANGE[0]) & (out_lon <= LON_RANGE[1]) & (out_lat >= LAT_RANGE[0]) & (out_lat <= LAT_RANGE[1])
            valid_index = x.loc[valid].index
            lon.loc[valid_index[ok]] = out_lon[ok]
            lat.loc[valid_index[ok]] = out_lat[ok]
        except Exception:
            pass
    return lat, lon

def standard_rows(folder, path, df):
    source_major = source_major_from_filename(path)
    major = major_from_filename(path)
    middle = middle_category(source_major, df)
    if source_major == 'school':
        school_level = middle.astype('string').str.strip()
        unsupported = school_level.notna() & school_level.ne('각종학교') & ~school_level.isin(SCHOOL_MAJOR_BY_LEVEL)
        if unsupported.any():
            values = sorted(school_level.loc[unsupported].dropna().unique().tolist())
            raise ValueError(f'정의되지 않은 학교급이 있습니다: {values}')
        output_major = school_level.map(SCHOOL_MAJOR_BY_LEVEL)
        keep = output_major.notna()
    else:
        output_major = pd.Series(major, index=df.index, dtype='object')
        keep = pd.Series(True, index=df.index)
    names = col(df, *NAME[source_major]).map(clean_text)
    roads = col(df, *ROAD[source_major]).map(clean_text)
    parcels = col(df, *PARCEL[source_major]).map(clean_text) if source_major in PARCEL else pd.Series(pd.NA, index=df.index)
    if source_major in TRANSFORMERS:
        lat, lon = projected_to_wgs84(df, source_major)
    else:
        lat = pd.to_numeric(col(df, *LAT[source_major]), errors='coerce') if source_major in LAT else pd.Series(np.nan, index=df.index)
        lon = pd.to_numeric(col(df, *LON[source_major]), errors='coerce') if source_major in LON else pd.Series(np.nan, index=df.index)
    result = pd.DataFrame({'생활지수': folder, '대분류': output_major, '중분류': middle, 'name': names, '도로명주소': roads, '지번주소': parcels, '위도': lat, '경도': lon, 'source_file': path.name}, index=df.index)
    return result.loc[keep, OUTPUT_COLUMNS]

def cultural_catalog_rows(path, sheet_name, name_column, middle_label):
    # 총람 파일은 4번째 행이 실제 컬럼명이므로 header=3으로 읽습니다.
    frame = pd.read_excel(path, sheet_name=sheet_name, header=3)
    frame.columns = [normalize_header(column) for column in frame.columns]
    required = {'시군구', name_column, '주소'}
    missing = sorted(required - set(frame.columns))
    if missing:
        raise ValueError(f'{sheet_name} 시트에서 필요한 컬럼이 없습니다: {missing}')

    region = (
        frame['시군구'].map(clean_text).astype('string').fillna('')
        .str.replace(r'\s+', '', regex=True)
    )
    selected = frame.loc[region.str.startswith('천안시', na=False)].copy()
    if sheet_name == '문예회관':
        selected = selected.loc[
            selected[name_column].map(clean_text).astype('string').fillna('').ne('천안예술의전당')
        ].copy()

    return pd.DataFrame(
        {
            '생활지수': 'leisure',
            '대분류': 'cultural_area',
            '중분류': 'cultural_facility',
            'name': selected[name_column].map(clean_text),
            '도로명주소': selected['주소'].map(clean_text),
            '지번주소': pd.Series(pd.NA, index=selected.index, dtype='object'),
            '위도': pd.Series(np.nan, index=selected.index, dtype=float),
            '경도': pd.Series(np.nan, index=selected.index, dtype=float),
            'source_file': f'{path.name}::{sheet_name}',
        },
        index=selected.index,
    )[OUTPUT_COLUMNS].reset_index(drop=True)

frames = []
source_stats = []
for folder in INDEX_FOLDERS:
    folder_path = ROOT / folder
    for path in sorted(folder_path.iterdir()):
        if path.suffix.lower() not in SOURCE_EXTENSIONS:
            continue
        # 공원은 현재 leisure 폴더에서 읽어 여가지수에 포함합니다.
        # 총람 파일도 leisure 폴더의 다른 원자료와 동일하게 이 반복문에서 읽습니다.
        if folder == 'leisure' and path.name == 'leisure_cultural_facilities_cheonan.xlsx':
            for sheet_name, name_column, middle_label in [
                ('문예회관', '시설명', '문예회관'),
                ('지방문화원', '문화원명', '지방문화원'),
            ]:
                catalog_frame = cultural_catalog_rows(
                    path, sheet_name, name_column, middle_label
                )
                frames.append(catalog_frame)
                source_stats.append({
                    '생활지수': 'leisure',
                    'source_file': f'{path.name}::{sheet_name}',
                    'source_rows': len(catalog_frame),
                    'source_format': f'excel:{sheet_name}',
                })
            continue
        df_source, source_format = read_tabular_with_metadata(path)
        frames.append(standard_rows(folder, path, df_source))
        source_stats.append({'생활지수': folder, 'source_file': path.name, 'source_rows': len(df_source), 'source_format': source_format})

if not frames:
    raise RuntimeError('통합할 CSV가 없습니다.')
integrated = pd.concat(frames, ignore_index=True)
integrated.to_csv(INTEGRATED_PATH, index=False, encoding='utf-8-sig', na_rep='')
print('통합파일 생성:', INTEGRATED_PATH.resolve(), '행 수:', len(integrated))

통합파일 생성: C:\Users\심현석\Documents\test\Cheonan-0825\grid_inputs\cheonan_all_facilities_integrated.csv 행 수: 2894


## 2. GIMI9 지오코딩 및 행정동 경계 검증

In [3]:
def normalize_address(value):
    return '' if pd.isna(value) else re.sub(r'\s+', ' ', str(value).strip())

def request_geocoding(addresses):
    response = requests.post(API_URL, headers={'Authorization': API_TOKEN, 'Accept': 'application/json', 'Content-Type': 'application/json'}, json={'q': addresses}, timeout=60)
    if response.status_code == 401:
        raise PermissionError('GIMI9 API 인증 실패(401): 토큰이 유효한지, 재발급된 토큰은 아닌지 확인하세요.')
    response.raise_for_status()
    results = response.json().get('results', [])
    if not isinstance(results, list):
        raise ValueError('GIMI9 응답 results가 list가 아닙니다.')
    return results

def dbf_records(raw):
    nrec, header_len, rec_len = struct.unpack('<I', raw[4:8])[0], struct.unpack('<H', raw[8:10])[0], struct.unpack('<H', raw[10:12])[0]
    fields, offset = [], 32
    while raw[offset] != 0x0D:
        desc = raw[offset:offset + 32]
        fields.append((desc[:11].split(b'\x00', 1)[0].decode('ascii', 'ignore'), desc[16]))
        offset += 32
    rows = []
    for i in range(nrec):
        row, pos, values = raw[header_len + i * rec_len:header_len + (i + 1) * rec_len], 1, {}
        for name, size in fields:
            values[name] = row[pos:pos + size].decode('cp949', 'ignore').strip(); pos += size
        rows.append(values)
    return rows

def read_shapes(path):
    with zipfile.ZipFile(path) as zf:
        shp = zf.read(next(n for n in zf.namelist() if n.lower().endswith('.shp')))
        attrs = dbf_records(zf.read(next(n for n in zf.namelist() if n.lower().endswith('.dbf'))))
    shapes, offset = [], 100
    while offset + 8 <= len(shp):
        record_no, words = struct.unpack('>2i', shp[offset:offset + 8]); offset += 8
        content = memoryview(shp)[offset:offset + words * 2]; offset += words * 2
        if len(content) < 44 or struct.unpack('<i', content[0:4])[0] != 5: continue
        bbox = struct.unpack('<4d', content[4:36]); parts_n, points_n = struct.unpack('<2i', content[36:44])
        part_start = 44; parts = list(struct.unpack('<' + 'i' * parts_n, content[part_start:part_start + 4 * parts_n])); point_start = part_start + 4 * parts_n
        points = [struct.unpack('<2d', content[point_start + 16 * i:point_start + 16 * (i + 1)]) for i in range(points_n)]
        rings = []
        for i, start in enumerate(parts):
            ring = points[start:(parts[i + 1] if i + 1 < len(parts) else points_n)]
            if len(ring) >= 3: rings.append((ring, (min(x for x, y in ring), min(y for x, y in ring), max(x for x, y in ring), max(y for x, y in ring))))
        if 0 < record_no <= len(attrs): shapes.append({'attrs': attrs[record_no - 1], 'bbox': bbox, 'rings': rings})
    return shapes

def point_in_ring(x, y, ring):
    inside, j = False, len(ring) - 1
    for i, (xi, yi) in enumerate(ring):
        xj, yj = ring[j]
        if (yi > y) != (yj > y) and x < (xj - xi) * (y - yi) / ((yj - yi) or 1e-30) + xi: inside = not inside
        j = i
    return inside

def point_in_shape(x, y, shape):
    if not (shape['bbox'][0] <= x <= shape['bbox'][2] and shape['bbox'][1] <= y <= shape['bbox'][3]): return False
    inside = False
    for ring, bbox in shape['rings']:
        if bbox[0] <= x <= bbox[2] and bbox[1] <= y <= bbox[3] and point_in_ring(x, y, ring): inside = not inside
    return inside

TO_BOUNDARY = Transformer.from_crs('EPSG:4326', 'EPSG:5186', always_xy=True)
def find_admin(lon, lat, shapes):
    x, y = TO_BOUNDARY.transform(lon, lat)
    return next((shape['attrs'] for shape in shapes if point_in_shape(x, y, shape)), None)

def result_record(item):
    success = item.get('success') is True
    lon = pd.to_numeric(item.get('x_axis'), errors='coerce') if success else np.nan
    lat = pd.to_numeric(item.get('y_axis'), errors='coerce') if success else np.nan
    valid = bool(success and pd.notna(lon) and pd.notna(lat) and -180 <= float(lon) <= 180 and -90 <= float(lat) <= 90)
    return {'latitude': float(lat) if valid else None, 'longitude': float(lon) if valid else None, 'geocode_success': success, 'geocode_coordinate_valid': valid, 'geocode_inputaddr': item.get('inputaddr', ''), 'geocode_error': item.get('errmsg', ''), 'geocode_address_cls': item.get('addressCls', ''), 'geocode_postcode': item.get('z', ''), 'geocode_h1_nm': item.get('h1_nm', ''), 'geocode_h23_nm': item.get('h23_nm', ''), 'geocode_hd_nm': item.get('hd_nm', ''), 'geocode_road_name': item.get('rm', ''), 'geocode_h1_cd': item.get('h1_cd', ''), 'geocode_h23_cd': item.get('h23_cd', ''), 'geocode_hd_cd': item.get('hd_cd', ''), 'geocode_kostat_h1_cd': item.get('kostat_h1_cd', ''), 'geocode_kostat_h2_cd': item.get('kostat_h2_cd', ''), 'geocode_bld_mgt_no': item.get('bld_mgt_no', '')}

df = integrated.copy()
original_columns = df.columns.tolist()
shapes = read_shapes(BOUNDARY_ZIP)
target_mask = df['위도'].isna() | df['경도'].isna()
target = df.loc[target_mask].copy()
target['geocode_query'] = target['도로명주소'].fillna(target['지번주소']).map(normalize_address)
target = target[target['geocode_query'].ne('')]
addresses = target['geocode_query'].drop_duplicates().tolist()
records_by_address, failed_batches, unmatched = {}, 0, 0
for start in range(0, len(addresses), BATCH_SIZE):
    batch = addresses[start:start + BATCH_SIZE]
    try: results = request_geocoding(batch)
    except PermissionError:
        raise
    except Exception as exc:
        failed_batches += 1
        print(f'배치 요청 실패: {start}~{start + len(batch) - 1} / {exc}')
        continue
    response_map = {normalize_address(item.get('inputaddr')): item for item in results if normalize_address(item.get('inputaddr'))}
    for address in batch:
        item = response_map.get(address)
        if item is None: unmatched += 1; continue
        record = result_record(item)
        admin = find_admin(record['longitude'], record['latitude'], shapes) if record['geocode_coordinate_valid'] else None
        record.update({'geocode_in_cheonan': admin is not None, 'geocode_boundary_adm_cd': (admin or {}).get('ADM_CD', ''), 'geocode_boundary_adm_nm': (admin or {}).get('ADM_NM', '')})
        record['geocode_accepted'] = bool(record['geocode_success'] and record['geocode_coordinate_valid'] and record['geocode_in_cheonan'])
        records_by_address[address] = record
    time.sleep(REQUEST_INTERVAL_SECONDS)
if addresses and not records_by_address:
    raise RuntimeError('지오코딩 결과가 없습니다. API 인증·네트워크·요청 주소를 확인하세요.')

audit_columns = ['geocode_success', 'geocode_coordinate_valid', 'geocode_inputaddr', 'geocode_error', 'geocode_address_cls', 'geocode_postcode', 'geocode_h1_nm', 'geocode_h23_nm', 'geocode_hd_nm', 'geocode_road_name', 'geocode_h1_cd', 'geocode_h23_cd', 'geocode_hd_cd', 'geocode_kostat_h1_cd', 'geocode_kostat_h2_cd', 'geocode_bld_mgt_no', 'geocode_in_cheonan', 'geocode_boundary_adm_cd', 'geocode_boundary_adm_nm', 'geocode_accepted']
for name in audit_columns: df[name] = pd.NA
df['geocode_query'] = df['도로명주소'].fillna(df['지번주소']).map(normalize_address)
filled_rows = 0
for index in df.index[target_mask]:
    record = records_by_address.get(df.at[index, 'geocode_query'])
    if record is None: continue
    for name in audit_columns: df.at[index, name] = record.get(name, pd.NA)
    if record['geocode_accepted']:
        if pd.isna(df.at[index, '위도']): df.at[index, '위도'] = record['latitude']
        if pd.isna(df.at[index, '경도']): df.at[index, '경도'] = record['longitude']
        filled_rows += 1
df = df.drop(columns=['geocode_query'])
df.to_csv(OUTPUT_PATH, index=False, encoding='utf-8-sig', na_rep='')
result = pd.read_csv(OUTPUT_PATH, encoding='utf-8-sig')
assert len(result) == len(integrated)
assert result[original_columns].columns.tolist() == original_columns
complete = integrated['위도'].notna() & integrated['경도'].notna()
# CSV 저장 전후의 정수·실수 자료형 차이는 허용하고 좌표값 자체만 비교합니다.
for coordinate in ['위도', '경도']:
    before = pd.to_numeric(integrated.loc[complete, coordinate], errors='coerce').to_numpy(dtype=float)
    after = pd.to_numeric(result.loc[complete, coordinate], errors='coerce').to_numpy(dtype=float)
    assert np.allclose(before, after, equal_nan=True, rtol=0, atol=1e-10)
print('최종 파일:', OUTPUT_PATH.resolve())
print('전체 행 수:', len(result))
print('지오코딩 대상 행 수:', int(target_mask.sum()))
print('API 조회 고유 주소 수:', len(addresses))
print('좌표 입력 행 수:', filled_rows)
print('최종 위도 결측 행 수:', int(result['위도'].isna().sum()))
print('최종 경도 결측 행 수:', int(result['경도'].isna().sum()))
print('실패 배치 수:', failed_batches, '미매칭 주소 수:', unmatched)

최종 파일: C:\Users\심현석\Documents\test\Cheonan-0825\grid_inputs\cheonan_all_facilities_geocoded.csv
전체 행 수: 2894
지오코딩 대상 행 수: 593
API 조회 고유 주소 수: 529
좌표 입력 행 수: 588
최종 위도 결측 행 수: 5
최종 경도 결측 행 수: 5
실패 배치 수: 0 미매칭 주소 수: 0
